# Executive Hotel Booking EDA

Complete end-to-end EDA workflow for the supplied hotel booking dataset.

**Raw data source:** GitHub repository `Divyanshi-1610/hotel-booking-eda`, file `data/uncleaned_hotel_bookings.csv`.

The notebook loads the raw CSV directly from GitHub, so no manual CSV upload is required after the repository is set up.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# GitHub repository settings
GITHUB_USER = "Divyanshi-1610"
GITHUB_REPO = "hotel-booking-eda"
GITHUB_BRANCH = "main"
GITHUB_CSV_PATH = "data/uncleaned_hotel_bookings.csv"

# The notebook automatically reads the RAW CSV from GitHub.
DATA_URL = f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}/{GITHUB_CSV_PATH}"
print("Raw dataset source:", DATA_URL)


In [ ]:
# Load the raw dataset directly from GitHub
df_raw = pd.read_csv(DATA_URL)
df = df_raw.copy()

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())


## Data quality audit


In [ ]:
quality=pd.DataFrame({"dtype":df.dtypes.astype(str),"missing_count":df.isna().sum(),"missing_pct":(df.isna().mean()*100).round(2),"unique_values":df.nunique()}).sort_values("missing_count",ascending=False)
display(quality)
print("Exact duplicates:",df.duplicated().sum())
display(df.describe(include="all").T)


## Cleaning and preprocessing


In [ ]:
clean=df.copy()
tokens=[""," ","NULL","null","N/A","NA","NaN","nan","None","none"]
for c in clean.select_dtypes("object").columns:
    clean[c]=clean[c].replace(tokens,np.nan)
    clean[c]=clean[c].apply(lambda x:x.strip() if isinstance(x,str) else x)
before=len(clean); clean=clean.drop_duplicates().reset_index(drop=True)
for c in clean.columns:
    if "date" in c.lower():
        x=pd.to_datetime(clean[c],errors="coerce")
        if x.notna().mean()>=.5: clean[c]=x
for c in clean.select_dtypes("object").columns:
    x=pd.to_numeric(clean[c],errors="coerce")
    if x.notna().mean()>=.9: clean[c]=x
for c in clean.select_dtypes(include=np.number).columns: clean[c]=clean[c].fillna(clean[c].median())
for c in clean.select_dtypes("object").columns:
    m=clean[c].mode()
    if len(m): clean[c]=clean[c].fillna(m.iloc[0])
print("Duplicates removed:",before-len(clean))
print("Remaining missing:",clean.isna().sum().sum())


## Feature engineering and outliers


In [ ]:
if {"stays_in_weekend_nights","stays_in_week_nights"}.issubset(clean.columns): clean["Total_Nights"]=clean.stays_in_weekend_nights+clean.stays_in_week_nights
if {"adults","children","babies"}.issubset(clean.columns): clean["Total_Guests"]=clean.adults+clean.children+clean.babies
if {"adr","Total_Nights","Total_Guests"}.issubset(clean.columns): clean["Estimated_Revenue"]=clean.adr*clean.Total_Nights*clean.Total_Guests
rows=[]
for c in clean.select_dtypes(include=np.number).columns:
    q1,q3=clean[c].quantile([.25,.75]); iqr=q3-q1
    n=0 if iqr==0 else ((clean[c]<q1-1.5*iqr)|(clean[c]>q3+1.5*iqr)).sum()
    rows.append([c,n,round(n/len(clean)*100,2)])
display(pd.DataFrame(rows,columns=["variable","outlier_count","outlier_pct"]).sort_values("outlier_count",ascending=False))


## Univariate analysis


In [ ]:
if "hotel" in clean: sns.countplot(data=clean,x="hotel"); plt.title("Bookings by Hotel Type"); plt.show()
if "is_canceled" in clean: sns.countplot(data=clean,x="is_canceled"); plt.title("Cancellation Distribution"); plt.show()
if "adr" in clean: sns.histplot(clean.adr,bins=50,kde=True); plt.title("ADR Distribution"); plt.show()


## Bivariate and group-wise analysis


In [ ]:
if {"hotel","is_canceled"}.issubset(clean.columns):
    x=clean.groupby("hotel").is_canceled.mean().mul(100); display(x.to_frame("Cancellation Rate (%)")); sns.barplot(x=x.index,y=x.values); plt.title("Cancellation Rate by Hotel Type"); plt.ylabel("Cancellation Rate (%)"); plt.show()
if {"market_segment","is_canceled"}.issubset(clean.columns):
    x=clean.groupby("market_segment").is_canceled.mean().mul(100).sort_values(ascending=False); display(x.to_frame("Cancellation Rate (%)")); sns.barplot(x=x.index,y=x.values); plt.xticks(rotation=35); plt.title("Cancellation Rate by Market Segment"); plt.show()
if {"hotel","adr"}.issubset(clean.columns):
    sns.boxplot(data=clean,x="hotel",y="adr"); plt.title("ADR by Hotel Type"); plt.show()


## Revenue, seasonality and correlation


In [ ]:
if {"Total_Nights","Estimated_Revenue"}.issubset(clean.columns):
    sns.scatterplot(data=clean,x="Total_Nights",y="Estimated_Revenue",alpha=.25); plt.title("Total Nights vs Estimated Revenue"); plt.show(); print("Correlation:",clean.Total_Nights.corr(clean.Estimated_Revenue))
if {"arrival_date_year","arrival_date_month"}.issubset(clean.columns):
    order=["January","February","March","April","May","June","July","August","September","October","November","December"]
    m=clean.groupby("arrival_date_month",observed=False).size().reindex(order); m.plot(marker="o",figsize=(12,5)); plt.title("Monthly Booking Volume"); plt.xticks(rotation=40); plt.show()
num=clean.select_dtypes(include=np.number); plt.figure(figsize=(14,10)); sns.heatmap(num.corr(),center=0,cmap="coolwarm"); plt.title("Correlation Heatmap"); plt.show()


## Key business insights
1. Cancellation behaviour is a major revenue risk and should be evaluated alongside booking volume.
2. Hotel type produces different pricing and cancellation patterns, supporting differentiated strategies.
3. Market segments can differ materially in cancellation behaviour, making channel-level controls useful.
4. Longer stays can increase booking revenue, supporting length-of-stay strategies.
5. ADR and demand patterns support seasonal and property-specific revenue management.

## Management recommendations
1. Introduce channel-specific cancellation and deposit policies.
2. Differentiate pricing by hotel type and demand conditions.
3. Promote longer stays through targeted packages.
4. Strengthen direct-booking incentives where intermediary cancellations are high.
5. Use seasonal pricing and targeted off-peak promotions.
6. Monitor cancellation, lead time, deposit and realised-revenue KPIs.
7. Build a cancellation-adjusted revenue dashboard.


In [ ]:
clean.to_csv("hotel_bookings_cleaned.csv",index=False)
print("Cleaned dataset saved as hotel_bookings_cleaned.csv")


## Conclusion
The analysis combines data-quality assessment, preprocessing, descriptive statistics, group-wise analysis, correlation analysis and visualisation to identify commercially useful patterns in hotel bookings.